# V4 - Target 0.85
**Improvements:**
- EfficientNet-B2 (better features)
- 5-crop TTA at inference
- Tempo augmentation
- Higher noise levels
- More training samples

In [79]:
!pip install -q librosa timm audiomentations

import os, glob, random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 11.4 MB/s eta 0:00:00
Device: cuda


In [80]:
# Config - AGGRESSIVE
CONFIG = {
    'sr': 22050,
    'duration': 8,  # Shorter for more variation
    'n_mels': 128,
    'n_fft': 2048,
    'hop_length': 512,
    'num_classes': 10,
    'batch_size': 24,
    'epochs': 10,
    'lr': 5e-4,
    'noise_prob': 0.9,
    'noise_level': (0.15, 0.5),  # Higher noise
    'cross_song_prob': 0.9,
    'tempo_prob': 0.5,  # NEW: tempo changes
    'tta_crops': 5,  # NEW: test-time augmentation
}

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 
          'jazz', 'metal', 'pop', 'reggae', 'rock']
genre_to_idx = {g: i for i, g in enumerate(GENRES)}
STEMS = ['drums', 'vocals', 'bass', 'other']

BASE_PATH = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR = os.path.join(BASE_PATH, 'genres_stems')
NOISE_DIR = os.path.join(BASE_PATH, 'ESC-50-master', 'audio')
TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
SAMPLE_SUB = os.path.join(BASE_PATH, 'sample_submission.csv')

In [81]:
# Load data
songs_by_genre = {g: [] for g in GENRES}
for genre in GENRES:
    genre_dir = os.path.join(STEMS_DIR, genre)
    if os.path.exists(genre_dir):
        for song_id in os.listdir(genre_dir):
            song_path = os.path.join(genre_dir, song_id)
            if os.path.isdir(song_path):
                if all(os.path.exists(os.path.join(song_path, f"{s}.wav")) for s in STEMS):
                    songs_by_genre[genre].append(song_path)

for g, songs in songs_by_genre.items():
    print(f"{g}: {len(songs)}")

noise_files = glob.glob(os.path.join(NOISE_DIR, '*.wav'))
print(f"Noise: {len(noise_files)}")

blues: 100
classical: 100
country: 100
disco: 100
hiphop: 100
jazz: 100
metal: 100
pop: 100
reggae: 100
rock: 100
Noise: 2000


In [82]:
# Audio functions with TEMPO augmentation
def load_stem(path, sr, duration):
    target_len = sr * duration
    try:
        audio, _ = librosa.load(path, sr=sr, duration=duration+2)  # Load extra for tempo
        if len(audio) < target_len:
            audio = np.tile(audio, 3)[:target_len]  # Loop short audio
        return audio[:target_len]
    except:
        return np.zeros(target_len)

def load_noise(sr, duration):
    if not noise_files:
        return np.zeros(sr * duration)
    path = random.choice(noise_files)
    audio = load_stem(path, sr, duration)
    return audio

def time_stretch(audio, rate):
    """Change tempo without changing pitch"""
    try:
        return librosa.effects.time_stretch(audio, rate=rate)
    except:
        return audio

def mix_stems_cross_song(genre, sr, duration, apply_tempo=False):
    """Mix stems from different songs with optional tempo variation"""
    target_len = sr * duration
    mixed = np.zeros(target_len, dtype=np.float32)
    genre_songs = songs_by_genre[genre]
    
    for stem in STEMS:
        song_path = random.choice(genre_songs)
        audio = load_stem(os.path.join(song_path, f"{stem}.wav"), sr, duration)
        
        # Random tempo change per stem (like test mashups)
        if apply_tempo and random.random() < 0.5:
            rate = random.uniform(0.9, 1.1)
            audio = time_stretch(audio, rate)
            if len(audio) < target_len:
                audio = np.pad(audio, (0, target_len - len(audio)))
            audio = audio[:target_len]
        
        mixed += audio
    return mixed

def add_noise(audio, level):
    noise = load_noise(CONFIG['sr'], CONFIG['duration'])
    if np.max(np.abs(noise)) > 0:
        noise = noise / np.max(np.abs(noise))
    return audio + level * noise

def normalize(audio):
    audio = audio - np.mean(audio)
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio)) * 0.9
    return np.clip(audio, -1, 1).astype(np.float32)

def to_mel(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=CONFIG['sr'], n_mels=CONFIG['n_mels'],
        n_fft=CONFIG['n_fft'], hop_length=CONFIG['hop_length']
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

In [83]:
# Dataset with aggressive augmentation
class TrainDataset(Dataset):
    def __init__(self, samples_per_genre=200):
        self.data = [(g, i) for g in GENRES for i in range(samples_per_genre)]
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        genre, _ = self.data[idx]
        label = genre_to_idx[genre]
        
        # Cross-song mixing with tempo
        apply_tempo = random.random() < CONFIG['tempo_prob']
        audio = mix_stems_cross_song(genre, CONFIG['sr'], CONFIG['duration'], apply_tempo)
        
        # Noise (high probability)
        if random.random() < CONFIG['noise_prob']:
            level = random.uniform(*CONFIG['noise_level'])
            audio = add_noise(audio, level)
        
        # Time shift
        if random.random() < 0.5:
            shift = random.randint(-CONFIG['sr']//2, CONFIG['sr']//2)
            audio = np.roll(audio, shift)
        
        # Random gain
        audio = audio * random.uniform(0.7, 1.3)
        
        audio = normalize(audio)
        mel = to_mel(audio)
        
        # SpecAugment (aggressive)
        for _ in range(2):
            if random.random() < 0.5:
                t = random.randint(0, 30)
                t0 = random.randint(0, max(1, mel.shape[1] - t - 1))
                mel[:, t0:t0+t] = 0
            if random.random() < 0.5:
                f = random.randint(0, 20)
                f0 = random.randint(0, max(1, mel.shape[0] - f - 1))
                mel[f0:f0+f, :] = 0
        
        mel_t = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
        return mel_t, label

class ValDataset(Dataset):
    def __init__(self, samples_per_genre=20):
        self.data = [(g, i) for g in GENRES for i in range(samples_per_genre)]
        random.seed(42)  # Fixed for consistency
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        genre, _ = self.data[idx]
        label = genre_to_idx[genre]
        random.seed(idx + 1000)  # Reproducible
        audio = mix_stems_cross_song(genre, CONFIG['sr'], CONFIG['duration'], apply_tempo=True)
        audio = add_noise(audio, 0.3)  # Fixed noise level
        audio = normalize(audio)
        mel = to_mel(audio)
        mel_t = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
        return mel_t, label

In [84]:
# Create loaders
train_dataset = TrainDataset(samples_per_genre=200)
val_dataset = ValDataset(samples_per_genre=25)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], num_workers=2)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Train: 2000, Val: 250


In [85]:
# Model - EfficientNet-B2 (better than B0)
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b2', pretrained=True, num_classes=0)
        self.head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self.backbone.num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, CONFIG['num_classes'])
        )
    def forward(self, x):
        return self.head(self.backbone(x))

model = Model().to(device)
print(f"Model: EfficientNet-B2")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

Model: EfficientNet-B2
Params: 8,427,532


In [86]:
# Training with mixup
def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1-lam) * x[idx], y, y[idx], lam

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.02)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=1)

best_acc = 0
for epoch in range(CONFIG['epochs']):
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    for data, target in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        data, target = data.to(device), target.to(device)
        
        # Mixup
        data, y_a, y_b, lam = mixup(data, target)
        
        optimizer.zero_grad()
        output = model(data)
        loss = lam * criterion(output, y_a) + (1-lam) * criterion(output, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        train_correct += (lam * (output.argmax(1) == y_a).float() + 
                         (1-lam) * (output.argmax(1) == y_b).float()).sum().item()
        train_total += target.size(0)
    
    scheduler.step()
    
    # Validate
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            val_correct += (model(data).argmax(1) == target).sum().item()
            val_total += target.size(0)
    
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best.pth')
    
    print(f"Epoch {epoch+1}: Train={train_acc:.4f}, Val={val_acc:.4f}, Best={best_acc:.4f}")

Epoch 1: 100%|██████████| 84/84 [04:28<00:00,  3.20s/it]


Epoch 1: Train=0.3491, Val=0.5640, Best=0.5640


Epoch 2: 100%|██████████| 84/84 [04:31<00:00,  3.23s/it]


Epoch 2: Train=0.5188, Val=0.6480, Best=0.6480


Epoch 3: 100%|██████████| 84/84 [04:25<00:00,  3.17s/it]


...

Epoch 8: Train=0.6442, Val=0.8200, Best=0.8200


Epoch 9: 100%|██████████| 84/84 [04:19<00:00,  3.09s/it]


Epoch 9: Train=0.6752, Val=0.8560, Best=0.8560


Epoch 10: 100%|██████████| 84/84 [04:19<00:00,  3.09s/it]


Epoch 10: Train=0.7170, Val=0.8760, Best=0.8760


In [87]:
# TTA (Test-Time Augmentation) - 5 crops
def load_audio_full(path, sr):
    """Load full audio for TTA crops"""
    try:
        audio, _ = librosa.load(path, sr=sr)
        return audio
    except:
        return np.zeros(sr * 10)

def get_crops(audio, sr, duration, n_crops=5):
    """Get multiple crops from audio"""
    target_len = sr * duration
    total_len = len(audio)
    
    if total_len <= target_len:
        audio = np.tile(audio, 3)
        total_len = len(audio)
    
    crops = []
    # Start, 25%, 50%, 75%, end
    positions = [0, 0.25, 0.5, 0.75, 1.0]
    for pos in positions[:n_crops]:
        start = int((total_len - target_len) * pos)
        start = max(0, min(start, total_len - target_len))
        crop = audio[start:start + target_len]
        if len(crop) < target_len:
            crop = np.pad(crop, (0, target_len - len(crop)))
        crops.append(crop)
    return crops

def predict_with_tta(model, audio_path, n_crops=5):
    """Predict with TTA - average over crops"""
    audio = load_audio_full(audio_path, CONFIG['sr'])
    crops = get_crops(audio, CONFIG['sr'], CONFIG['duration'], n_crops)
    
    all_logits = []
    for crop in crops:
        crop = normalize(crop)
        mel = to_mel(crop)
        mel_t = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1)
        with torch.no_grad():
            logits = model(mel_t.to(device))
            all_logits.append(logits)
    
    # Average logits
    avg_logits = torch.stack(all_logits).mean(0)
    return avg_logits.argmax(1).item()

In [88]:
# Test with TTA
model.load_state_dict(torch.load('best.pth'))
model.eval()

test_df = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB)
id_to_file = dict(zip(test_df['id'], test_df['filename']))

test_files = [os.path.join(BASE_PATH, id_to_file[row['id']]) for _, row in sample_sub.iterrows()]
print(f"Test files: {len(test_files)}")
print(f"Using {CONFIG['tta_crops']}-crop TTA")

preds = []
for path in tqdm(test_files, desc="TTA Inference"):
    pred = predict_with_tta(model, path, CONFIG['tta_crops'])
    preds.append(pred)

submission = sample_sub.copy()
submission['genre'] = [GENRES[p] for p in preds]
submission.to_csv('submission.csv', index=False)
print("\nSubmission saved!")
print(submission['genre'].value_counts())

Test files: 3020
Using 5-crop TTA


TTA Inference: 100%|██████████| 3020/3020 [10:34<00:00,  4.76it/s]



Submission saved!
genre
pop          358
jazz         321
classical    307
reggae       302
metal        301
country      301
disco        288
hiphop       282
rock         280
blues        280
Name: count, dtype: int64


In [89]:
def tta_heavy(path, n=10):                                                                                                                  
  try:                                                                                                                                    
      audio_full, _ = librosa.load(path, sr=CONFIG['sr'])                                                                                 
  except:                                                                                                                                 
      audio_full = np.zeros(CONFIG['sr'] * 20)                                                                                            
                                                                                                                                          
  target = CONFIG['sr'] * CONFIG['duration']                                                                                              
  if len(audio_full) < target:                                                                                                            
      audio_full = np.tile(audio_full, 4)                                                                                                 
                                                                                                                                          
  probs = []                                                                                                                              
  positions = np.linspace(0, max(0, len(audio_full)-target), n).astype(int)                                                               
                                                                                                                                          
  for pos in positions:                                                                                                                   
      crop = audio_full[pos:pos+target]                                                                                                   
      if len(crop) < target:                                                                                                              
          crop = np.pad(crop, (0, target-len(crop)))                                                                                      
      crop = normalize(crop)                                                                                                              
      mel = to_mel(crop)                                                                                                                  
      mel_t = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).unsqueeze(0).repeat(1,3,1,1).to(device)                                 
      with torch.no_grad():                                                                                                               
          probs.append(torch.softmax(model(mel_t), dim=1))                                                                                
                                                                                                                                          
  return torch.stack(probs).mean(0)                                                                                                       
                                                                                                                                          
print("Running 10-crop TTA...")                                                                                                             
all_probs = []                                                                                                                              
for f in tqdm(test_files):                                                                                                                  
  all_probs.append(tta_heavy(f, n=10))                                                                                                    
                                                                                                                                          
probs_tensor = torch.stack(all_probs)                                                                                                       
preds = probs_tensor.argmax(2).squeeze().cpu().numpy()                                                                                      
                                                                                                                                          
submission = sample_sub.copy()                                                                                                              
submission['genre'] = [GENRES[p] for p in preds]                                                                                            
submission.to_csv('submission_90.csv', index=False)                                                                                         
print("Saved submission_90.csv")                                                                                                            
print(submission['genre'].value_counts())                                                                                                   
                                              

Running 10-crop TTA...


100%|██████████| 3020/3020 [18:57<00:00,  2.65it/s]

Saved submission_90.csv
genre
pop          364
jazz         325
country      319
classical    305
reggae       302
metal        297
disco        291
hiphop       277
rock         270
blues        270
Name: count, dtype: int64
